In this notebook we tried to do a scoped down version of weak to strong generalization on the basic OpenAI paper and this is an attempt in Paper2Code. We will be using the boolq dataset for this and our weak model will be the QN 2.5 1.5 billion parameters and the 7 billion parameter will be our stronger model. The idea the notebook is to replicate it to as much closer as possible and to get the PGR score that we can achieve

In [ ]:
import torch as t
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorWithPadding
from torch.utils.data import DataLoader
import tqdm
import gc
device = t.device("cuda" if t.cuda.is_available() else "cpu")
print(f"Using device: {device}")

weak_model_id = 'Qwen/Qwen2.5-1.5B'
strong_model_id = 'Qwen/Qwen2.5-7B'

dataset = load_dataset('google/boolq')
tokenizer = AutoTokenizer.from_pretrained(weak_model_id)
tokenizer.pad_token = tokenizer.eos_token

Using device: cpu


In [8]:
# Define integer labels for Ground Truth
def format_and_label(example):
    prompt = f"Passage: {example['passage']}\nQuestion: {example['question']}?\nAnswer:"
    label_int = 1 if example['answer'] else 0
    return {"prompt": prompt, "label_int": label_int}

def tokenize_func(example):
    return tokenizer(example["prompt"], truncation=True, max_length=512)

In [9]:
processed_dataset = dataset.map(format_and_label)
tokenized_dataset = processed_dataset.map(
    tokenize_func, 
    batched=True, 
    remove_columns=["passage", "question", "answer", "prompt"] 
)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label_int"])

# Split Dataset (BoolQ has ~9.4k train rows)
# 4,000 for Weak Teacher, 4,000 for Strong Student
ds_weak_train = tokenized_dataset['train'].select(range(0, 4000))
ds_strong_train = tokenized_dataset['train'].select(range(4000, 8000))
ds_test = tokenized_dataset['validation'] # ~3.2k rows for evaluation

collator = DataCollatorWithPadding(tokenizer=tokenizer)

# DataLoaders
dl_weak_train = DataLoader(ds_weak_train, batch_size=4, shuffle=True, collate_fn=collator)
dl_strong_inference = DataLoader(ds_strong_train, batch_size=4, shuffle=False, collate_fn=collator)
dl_test = DataLoader(ds_test, batch_size=4, shuffle=False, collate_fn=collator)

print("DataLoaders ready!")

DataLoaders ready!


In [10]:
def load_classification_model(model_id):
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    yes_id_token = tokenizer.encode("Yes", add_special_tokens=False)[0]
    no_id_token = tokenizer.encode("No", add_special_tokens=False)[0]

    hidden_size = model.config.hidden_size
    llm_head = model.lm_head

    with t.no_grad():
        yes_weights = llm_head.weight[yes_id_token] 
        no_weights = llm_head.weight[no_id_token]
    classification_head = t.nn.Linear(hidden_size, 2).to(device)
    with t.no_grad():
        classification_head.weight[0] = no_weights
        classification_head.weight[1] = yes_weights
    
    model.lm_head = classification_head
    model.config.vocab_size = 2
    return model

In [11]:
def evaluate_model(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with t.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label_int'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = t.argmax(logits, dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct, total

In [ ]:
model_weak = load_classification_model(weak_model_id)

print("Training Weak Teacher...")

optimizer = t.optim.AdamW(model_weak.parameters(), lr=5e-5)
criterion = t.nn.CrossEntropyLoss()

model_weak.train()

for epoch in range(1):
    progress_bar = tqdm(dl_weak_train, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label_int'].to(device)

        outputs = model_weak(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({"loss": loss.item()})

print("Evaluating Weak Teacher...")
correct, total = evaluate_model(model_weak, dl_test)
weak_teacher_accuracy = correct / total if total > 0 else 0
print(f"Weak Teacher Accuracy: {correct}/{total} = {weak_teacher_accuracy:.4f}")


In [ ]:
model_weak.eval()
weak_soft_labels = []

with t.no_grad():
    for batch in tqdm(dl_strong_inference, desc="Generating Soft Labels"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model_weak(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits[t.arange(logits.size(0)), attention_mask.sum(dim=1) - 1]
        probs = t.softmax(logits, dim=-1)
        weak_soft_labels.extend(probs.cpu().tolist())

ds_strong_train = ds_strong_train.add_column("soft_labels", weak_soft_labels)
ds_strong_train.set_format("torch", columns=["input_ids", "attention_mask", "label_int", "soft_labels"])

dl_strong_train = DataLoader(ds_strong_train, batch_size=4, shuffle=True, collate_fn=collator)

del model_weak
t.cuda.empty_cache()
gc.collect()


In [ ]:
# --- NOTEBOOK CELL 5: 7B Training Logic ---
def train_7b_model(model, dataloader, use_soft_labels=False):
    model.train()
    optimizer = t.optim.AdamW(model.parameters(), lr=1e-5)
    criterion = t.nn.CrossEntropyLoss()
    
    progress = tqdm(dataloader, desc="Training 7B")
    for batch in progress:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        
        # Select target type
        if use_soft_labels:
            targets = batch['soft_label'].to(model.device) # Float probabilities
        else:
            targets = batch['label_int'].to(model.device)  # Ground truth integers
            
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        seq_lengths = attention_mask.sum(dim=1) - 1
        logits = outputs.logits[t.arange(input_ids.size(0)), seq_lengths]
        
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        progress.set_postfix({'loss': f"{loss.item():.4f}"})

In [ ]:
# --- NOTEBOOK CELL 6: Train Strong Student on Weak Labels ---
model_student = load_classification_model(strong_model_id)

print("\nTraining 7B Student on WEAK LABELS...")
train_7b_model(model_student, dl_strong_train, use_soft_labels=True)

# Evaluate immediately
correct, total = evaluate_model(model_student, dl_test)
w2s_accuracy = correct / total if total > 0 else 0
print(f"Weak-to-Strong Accuracy: {w2s_accuracy * 100:.2f}%")

# Free VRAM
del model_student
t.cuda.empty_cache()
gc.collect()

In [ ]:
model_ceiling = load_classification_model(strong_model_id)

print("\nTraining 7B Ceiling on GROUND TRUTH...")
train_7b_model(model_ceiling, dl_strong_train, use_soft_labels=False)

# Evaluate immediately
correct, total = evaluate_model(model_ceiling, dl_test)
ceiling_accuracy = correct / total if total > 0 else 0
print(f"Ceiling Accuracy: {ceiling_accuracy * 100:.2f}%")

# Free VRAM
del model_ceiling
t.cuda.empty_cache()
gc.collect()

In [ ]:
print("\n=== FINAL EXPERIMENT RESULTS ===")
print(f"Weak Teacher (1.5B):       {weak_teacher_accuracy * 100:.2f}%")
print(f"Strong Student (7B + W2S): {w2s_accuracy * 100:.2f}%")
print(f"Ceiling Model (7B + GT):   {ceiling_accuracy * 100:.2f}%")
print("-" * 32)

performance_gap = ceiling_accuracy - weak_teacher_accuracy
recovered = w2s_accuracy - weak_teacher_accuracy

if performance_gap > 0:
    pgr = (recovered / performance_gap) * 100
    print(f"Performance Gap Recovered (PGR): {pgr:.2f}%")
else:
    print("Ceiling did not outperform Weak Teacher. Ensure training converged.")